## Memory optimizations - Referrence Implementation
For the bottlenecks you may have observed while profiling `debug-llm-bottlenecks-with-nvidia-nsight/lab-02/02-memory-optimization/ft-tinyllama-lora-bf16-baseline.ipynb`, current notebook has referrence optimizations implemented.
Similar to earlier notebook, we recommend you to just profile the last cell `lora_trainer.train()`, to get more focused report we will use for review.
Sample screenshots sowing improvements is attached to the end of this notebook.

Feel free to optimize the notebook further and experiment.

Also, refer to the training parameter referrence guide for details on the key parameters optimized in this notebook -
         `debug-llm-bottlenecks-with-nvidia-nsight/lab-02/02-memory-optimization/training_parameters_reference_guide.md`

In [ ]:
!pip install datasets transformers accelerate trl peft

In [ ]:
import torch
from datasets import Dataset, load_dataset
from peft import LoraConfig, AutoPeftModelForCausalLM
from transformers import AutoTokenizer, AutoModelForCausalLM, TrainingArguments, IntervalStrategy
from trl import SFTTrainer

dataset = "greenwich157/telco-5G-core-faults"
model_id= "TinyLlama/TinyLlama-1.1B-Chat-v1.0"

def prepare_training_data(dataset_id):
    data = load_dataset(dataset_id, split="train")
    data_df = data.to_pandas()
    data_df["text"] = data_df[["instruction", "input", "output"]].apply(lambda x:"<|system|>\n" + x.instruction + "</s><|user|>\n" + x.input + "</s>\n<|assistant|>\n" + x.output + "</s>", axis=1)
    data = Dataset.from_pandas(data_df)
    return data

full_train_data = prepare_training_data(dataset)
print ("training data prepared successfully ", full_train_data)
full_train_data = full_train_data.remove_columns(['instruction', 'input', 'output'])

full_train_data = full_train_data.select(range(240)) # trimming to reduce report size
full_train_data = full_train_data.train_test_split(test_size=0.20, seed=42)
print ("training data trimmed and split successfully ", full_train_data)
train_data = full_train_data['train']
test_data = full_train_data['test']

torch.cuda.empty_cache()

In [ ]:
lora_model = AutoModelForCausalLM.from_pretrained(
    model_id, 
    device_map="cuda",
    dtype=torch.bfloat16,
    attn_implementation="eager",
)

lora_train_args = TrainingArguments(
    bf16=True,
    batch_eval_metrics=False,
    dataloader_drop_last=True,
    dataloader_num_workers=4,
    dataloader_persistent_workers=True,
    dataloader_pin_memory=True,
    dataloader_prefetch_factor=2,
    do_eval=False,
    disable_tqdm=True,
    eval_accumulation_steps=0,
    eval_do_concat_batches=False,
    eval_strategy=IntervalStrategy.NO,
    full_determinism=False, 
    gradient_accumulation_steps=4,
    gradient_checkpointing=False,
    logging_strategy=IntervalStrategy.STEPS,
    logging_steps=10,
    learning_rate=2e-4,
    lr_scheduler_type="constant",
    max_grad_norm=None,
    num_train_epochs=2,
    optim="adamw_torch",
    output_dir="ft-tinyllama-lora-bf16-optimized",
    per_device_train_batch_size=24,
    per_device_eval_batch_size=16,
    prediction_loss_only=True,
    remove_unused_columns=True,
    skip_memory_metrics=True,
    save_strategy=IntervalStrategy.NO,
    # save_steps=50,
    # warmup_steps=0,
    warmup_ratio=0.1,
)
print ("training arguments initialized successfully ", lora_train_args)


peft_config = LoraConfig(
    lora_alpha=8, 
    lora_dropout=0.1, 
    r=4,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=["q_proj", "v_proj"],
)
# print ("peft_config created ", peft_config)

lora_trainer = SFTTrainer(
    model=lora_model,
    peft_config=peft_config,
    args=lora_train_args,
    train_dataset=train_data, 
    eval_dataset=test_data,
)
print ("Trainer class initialized and staring the PEFT fine-tuning")

In [ ]:
# import torch
# print(f"GPU 0 memory: {torch.cuda.get_device_properties(0).total_memory}")
# print(f"GPU 0 allocated: {torch.cuda.memory_allocated(0)}")
# torch.cuda.empty_cache()

In [ ]:
lora_trainer.train()
print ("PEFT LORA fine-tuning Completed Successfully")

## Referrence profiler screenshot after optimization

![](../../static-assets/lab-02/02-memory-optimization-fix-rep-1.png)